# Analyze ONNX for GPT

Analyze one NeuroGolf ONNX file and generate a compact text package that can be pasted into GPT for optimization ideas.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from neurogolf_score import score_model_file
from neurogolf_onnx_analysis import (
    compare_score_rows,
    compact_model_markdown,
    gpt_analysis_prompt,
    run_model_on_examples,
    summarize_model,
    summarize_task,
)

DATA_DIR = repo_root / "neurogolf-2026"
WORK_DIR = repo_root / "outputs" / "gpt_onnx_analysis"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TASK_ID = "task101"
ONNX_PATH = repo_root / "solution" / "7157.62" / f"{TASK_ID}.onnx"
CANDIDATE_ONNX_DIR = repo_root / "outputs" / "gpt_workbench" / TASK_ID
CANDIDATE_ONNX_PATHS = sorted(CANDIDATE_ONNX_DIR.glob("*.onnx"))

TASK_ID, ONNX_PATH, CANDIDATE_ONNX_DIR, CANDIDATE_ONNX_PATHS


('task101',
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6411.33/task101.onnx'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task101'),
 [PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_workbench/task101/424_107_template_bool_blue_offset_direct_o15_pt7_pl7.onnx')])

## Task Summary

In [442]:
task_summary = summarize_task(TASK_ID, DATA_DIR)

print(f"task    : {task_summary['task']}")
print(f"train   : {task_summary['num_train']}")
print(f"test    : {task_summary['num_test']}")
print(f"arc-gen : {task_summary['num_arc_gen']}")
print(f"colors  : {task_summary['color_counts']}")

try:
    import pandas as pd

    display(pd.DataFrame(task_summary["examples"]))
except ImportError:
    task_summary["examples"]

task    : task101
train   : 3
test    : 1
arc-gen : 262
colors  : {0: 95369, 1: 5673, 2: 4672}


,subset,index,input_shape,output_shape,input_cells,output_cells
0,train,0,14x12,14x12,168,168
1,train,1,14x12,14x12,168,168
2,train,2,14x12,14x12,168,168
3,test,0,17x21,17x21,357,357
4,arc-gen,0,17x14,17x14,238,238
...,...,...,...,...,...,...
261,arc-gen,257,14x12,14x12,168,168
262,arc-gen,258,14x12,14x12,168,168
263,arc-gen,259,14x12,14x12,168,168
264,arc-gen,260,14x12,14x12,168,168


## Score

In [443]:
base_score_row = score_model_file(ONNX_PATH, DATA_DIR)
candidate_score_rows = []

for candidate_path in CANDIDATE_ONNX_PATHS:
    row = score_model_file(candidate_path, DATA_DIR)
    row["candidate_file"] = candidate_path.name
    row["candidate_path"] = str(candidate_path)
    row.update(compare_score_rows(base_score_row, row))
    candidate_score_rows.append(row)

# Keep this alias for the GPT prompt generation cells below.
score_row = base_score_row

print("Base/reference ONNX:")
display(base_score_row)

print(f"Candidate ONNX files: {len(candidate_score_rows)}")
try:
    import pandas as pd

    if candidate_score_rows:
        score_cols = [
            "candidate_file", "status", "score", "score_delta", "cost", "cost_delta",
            "memory", "memory_delta", "params", "params_delta", "filesize", "filesize_delta", "error"
        ]
        display(pd.DataFrame(candidate_score_rows)[score_cols].sort_values("score", ascending=False).reset_index(drop=True))
    else:
        print(f"No candidate ONNX files found in {CANDIDATE_ONNX_DIR}")
except ImportError:
    candidate_score_rows


Base/reference ONNX:


{'task': 'task101',
 'candidate': '6411.33',
 'file': '/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6411.33/task101.onnx',
 'filesize': 32954,
 'status': 'ok',
 'memory': 154080,
 'params': 2968,
 'cost': 157048.0,
 'score': 13.03569322991213,
 'error': ''}

Candidate ONNX files: 1


,candidate_file,status,score,score_delta,cost,cost_delta,memory,memory_delta,params,params_delta,filesize,filesize_delta,error
0,424_107_template_bool_blue_offset_direct_o15_p...,ok,13.331842,0.296148,116793.0,-40255.0,113826,-40254,2967,-1,27538,-5416,


## Candidate Comparison

In [444]:
base_validation = run_model_on_examples(ONNX_PATH, TASK_ID, DATA_DIR)
candidate_validation_rows = []
candidate_validations = {}

for candidate_path in CANDIDATE_ONNX_PATHS:
    report = run_model_on_examples(candidate_path, TASK_ID, DATA_DIR)
    candidate_validations[candidate_path.name] = report
    counts = report.get("counts", {})
    candidate_validation_rows.append({
        "candidate_file": candidate_path.name,
        "validation_status": report.get("status"),
        "pass": counts.get("pass", 0),
        "fail": counts.get("fail", 0),
        "error": counts.get("error", 0),
        "skipped_large_grid": counts.get("skipped_large_grid", 0),
        "first_failure": report.get("first_failure"),
    })

print("Base validation:")
display({k: v for k, v in base_validation.items() if k != "rows"})

print("Candidate validations:")
try:
    import pandas as pd

    validation_df = pd.DataFrame(candidate_validation_rows)
    if candidate_score_rows:
        score_df = pd.DataFrame(candidate_score_rows)
        compare_df = validation_df.merge(
            score_df[["candidate_file", "status", "score", "score_delta", "cost_delta", "memory_delta", "params_delta", "filesize_delta"]],
            on="candidate_file",
            how="left",
        )
        display(compare_df.sort_values(["validation_status", "score_delta"], ascending=[True, False]).reset_index(drop=True))
    else:
        display(validation_df)
except ImportError:
    candidate_validation_rows


Base validation:


{'status': 'ok', 'counts': {'pass': 266}, 'first_failure': None}

Candidate validations:


,candidate_file,validation_status,pass,fail,error,skipped_large_grid,first_failure,status,score,score_delta,cost_delta,memory_delta,params_delta,filesize_delta
0,424_107_template_bool_blue_offset_direct_o15_p...,ok,266,0,0,0,None,ok,13.331842,0.296148,-40255.0,-40254,-1,-5416


In [445]:
valid_improvements = []
for row in candidate_score_rows:
    validation = candidate_validations.get(row["candidate_file"], {})
    if validation.get("status") == "ok" and row.get("score_delta", 0) > 0:
        valid_improvements.append(row)

print(f"Valid score improvements: {len(valid_improvements)}")
try:
    import pandas as pd

    if valid_improvements:
        display(pd.DataFrame(valid_improvements).sort_values("score_delta", ascending=False).reset_index(drop=True))
except ImportError:
    valid_improvements


Valid score improvements: 1


,task,candidate,file,filesize,status,memory,params,cost,score,error,candidate_file,candidate_path,filesize_delta,memory_delta,params_delta,cost_delta,score_delta,base_status,candidate_status
0,424_107_template_bool_blue_offset_direct_o15_p...,task101,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,27538,ok,113826,2967,116793.0,13.331842,,424_107_template_bool_blue_offset_direct_o15_p...,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,-5416,-40254,-1,-40255.0,0.296148,ok,ok


In [446]:
failed_candidates = {
    name: report.get("first_failure")
    for name, report in candidate_validations.items()
    if report.get("first_failure")
}
failed_candidates


{}

## ONNX Structure

In [447]:
model_summary = summarize_model(ONNX_PATH)

print(f"file         : {model_summary['filename']}")
print(f"filesize     : {model_summary['filesize']}")
print(f"ir_version   : {model_summary['ir_version']}")
print(f"opset_imports: {model_summary['opset_imports']}")
print(f"inputs       : {model_summary['inputs']}")
print(f"outputs      : {model_summary['outputs']}")
print(f"nodes        : {model_summary['num_nodes']}")
print(f"initializers : {model_summary['num_initializers']}")
print(f"op_counts    : {model_summary['op_counts']}")

file         : task101.onnx
filesize     : 32954
ir_version   : 10
opset_imports: [{'domain': '', 'version': 11}]
inputs       : [{'name': 'input', 'elem_type': 'FLOAT', 'shape': [1, 10, 30, 30]}]
outputs      : [{'name': 'output', 'elem_type': 'FLOAT', 'shape': [1, 10, 30, 30]}]
nodes        : 109
initializers : 37
op_counts    : {'Add': 8, 'ArgMax': 2, 'Cast': 20, 'Concat': 1, 'Conv': 17, 'ConvTranspose': 2, 'Div': 2, 'Equal': 15, 'Mul': 19, 'Reshape': 2, 'ScatterElements': 1, 'Sign': 5, 'Slice': 5, 'Sub': 6, 'Sum': 4}


## Initializers

In [448]:
try:
    import pandas as pd

    display(pd.DataFrame(model_summary["initializers"]))
except ImportError:
    model_summary["initializers"]

,name,dtype,dims,numel,bytes,sample
0,starts_0,INT64,[1],1,8,[0]
1,ends_2,INT64,[1],1,8,[2]
2,ends_3,INT64,[1],1,8,[3]
3,axes_1,INT64,[1],1,8,[1]
4,cross_kernel,FLOAT16,"[1, 1, 3, 3]",9,18,"[0.0, 0.39990234375, 0.0, 0.39990234375, 0.399..."
5,one_f16,FLOAT16,"[1, 1, 1, 1]",1,2,[1.0]
6,flat_shape,INT64,[4],4,32,"[1, 1, 1, 900]"
7,idx_range_int64,INT64,"[1, 1, 1, 900]",900,7200,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]"
8,shape_30_30,INT64,[4],4,32,"[1, 1, 30, 30]"
9,W_int64,INT64,"[1, 1, 1, 1]",1,8,[30]


## Nodes

In [449]:
try:
    import pandas as pd

    nodes_df = pd.DataFrame(model_summary["nodes"])
    display(nodes_df[["index", "op_type", "name", "inputs", "outputs", "attributes"]])
except ImportError:
    model_summary["nodes"][:20]

,index,op_type,name,inputs,outputs,attributes
0,0,Slice,mask_blue_f32,"[input, axes_1, ends_2, axes_1]",[mask_blue_f32],{}
1,1,Cast,mask_blue,[mask_blue_f32],[mask_blue],{'to': 10}
2,2,Slice,mask_red_f32,"[input, ends_2, ends_3, axes_1]",[mask_red_f32],{}
3,3,Cast,mask_red,[mask_red_f32],[mask_red],{'to': 10}
4,4,Add,valid,"[mask_blue, mask_red]",[valid],{}
...,...,...,...,...,...,...
104,104,Add,new_blue_corrected,"[mask_blue, painted_blue_on_black]",[new_blue_corrected],{}
105,105,Sub,new_black_corrected,"[mask_black, painted_blue_on_black]",[new_black_corrected],{}
106,106,Concat,task101_update_concat_f16,"[new_black_corrected, new_blue_corrected]",[task101_update_f16_compact],{'axis': 1}
107,107,Cast,task101_update_cast_f32,[task101_update_f16_compact],[update_f32],{'to': 1}


## GPT Package

In [450]:
model_markdown = compact_model_markdown(model_summary, score_row=score_row, max_nodes=160)
prompt = gpt_analysis_prompt(task_summary, model_markdown)

prompt_path = WORK_DIR / f"{TASK_ID}_gpt_analysis_prompt.md"
prompt_path.write_text(prompt)

print(f"Saved GPT prompt: {prompt_path}")
print(f"Prompt characters: {len(prompt)}")

Saved GPT prompt: /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/gpt_onnx_analysis/task101_gpt_analysis_prompt.md
Prompt characters: 52363


In [451]:
print(prompt)

You are helping optimize a NeuroGolf ONNX model.

Rules:
- Input tensor: float32 [1, 10, 30, 30], name 'input'.
- Output tensor: float32 [1, 10, 30, 30], name 'output'.
- Static tensor shapes only.
- One input and one output only.
- Banned ops: Loop, Scan, NonZero, Unique, Script, Function, Compress.
- Optimize score by reducing memory + params.
- Assume public examples pass unless told otherwise.

Requested output:
1. Summarize what the current ONNX appears to do.
2. Identify redundant or expensive parts.
3. Propose safe rewrite candidates ranked by risk and likely score gain.
4. Do not write code yet.

Task summary:
- task: task101
- train examples: 3
- test examples: 1
- arc-gen examples: 262
- color counts: {0: 95369, 1: 5673, 2: 4672}
- example shapes: [{'subset': 'train', 'index': 0, 'input_shape': '14x12', 'output_shape': '14x12', 'input_cells': 168, 'output_cells': 168}, {'subset': 'train', 'index': 1, 'input_shape': '14x12', 'output_shape': '14x12', 'input_cells': 168, 'output

Suggested workflow: paste the generated prompt into GPT first and ask for analysis only. After choosing a candidate rewrite, ask GPT to produce a `build_taskXXX.py` script that creates a new ONNX.